In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import math
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# download pre-trained model
pretrained_weights = ViT_B_16_Weights.IMAGENET1K_V1
model = vit_b_16(weights=pretrained_weights).to(device)

In [ ]:
# resolution of the model input
sz = pretrained_weights.transforms().crop_size
size = (sz, sz)
print(size)

In [ ]:
# adjust the model for 10 categories of MNIST dataset
pretrained_weights = ViT_B_16_Weights.IMAGENET1K_V1
model = vit_b_16(weights=pretrained_weights).to(device)
num_classes = 10
model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
model = model.to(device)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=pretrained_weights.transforms().mean,
        std=pretrained_weights.transforms().std
    ),
])

In [ ]:
# get datasets
train_dataset = datasets.MNIST(root='data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='data', train=False, download=True, transform=transform)

In [ ]:
# create train and test dataloader
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
def train(num_epochs):
    global epochs
    for _ in range(num_epochs):
        epochs += 1
        # train
        model.train()
        total_loss = 0
        correct = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
        train_loss = total_loss / len(train_loader)
        train_acc = correct / len(train_loader.dataset)
        # test
        model.eval()
        correct = 0
        for images, labels in test_loader:
            with torch.no_grad():
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
        test_acc = correct / len(test_loader.dataset)
        print(f"Epoch [{epochs}] Loss: {train_loss:.4f} Train Acc: {train_acc:.4f} Test Acc: {test_acc:.4f}")

In [ ]:
# training
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.heads.head.parameters(), lr=0.001)
epochs = 0
for param in model.parameters():
    param.requires_grad = False
for param in model.heads.head.parameters():
    param.requires_grad = True
train(5)

In [ ]:
# fine tuning
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 0
for param in model.parameters():
    param.requires_grad = True
train(15)

In [ ]:
model.eval()

In [ ]:
# get few samples
test_iter = iter(test_loader)
x_sample, y_sample = next(test_iter)
print(x_sample.shape)

In [ ]:
# use the model on the samples
input_images = x_sample[0:100].to(device)
output_logits = model(input_images)
output_categories = [ category.item() for category in torch.argmax(output_logits.to('cpu'), dim=1) ]
print(output_categories)
categories = [ category.item() for category in y_sample ]
print(categories)

In [ ]:
# show results
def render(digit):
    img = np.zeros((28, 28), dtype=np.uint8)
    cv2.putText(img, str(digit), (6, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.9, 255, 2)
    return img

plt.figure(figsize=(20, 40))
for b in range(6):
    for i in range(10):
        input_img = (x_sample[10*b+i].mean(dim=0).detach().numpy()*255).astype(np.uint8)
        plt.subplot(20, 10, 20*b+i+1)
        plt.imshow(cv2.resize(input_img,(28,28)), cmap='gray')
        plt.axis('off')
        output_img = render(output_categories[10*b+i])
        plt.subplot(20, 10, 20*b+i+1+10)
        plt.imshow(~output_img, cmap='gray')
        plt.axis('off')
plt.show()